In [16]:
import pandas as pd
import numpy as np  

# Leer el dataset sucio

df = pd.read_csv("sp500_financials.csv", encoding="utf-8")

# Ver las primeras filas del dataset 
df.head(10)

# Informacion completa del dataser

print("=" * 60)
print("Exploracion del dataset sS&p 500 Financials")
print("=" * 60)

print(f"\n📊 dimensiones: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"\n📋 Columnas:\n{list(df.columns)}")

print("\n" + "=" * 60 )
print("Tipos de datos")
print("=" * 60)
print(df.dtypes)    

print("\n" + "=" * 60)
print("VALORES FALTANTES (NaN) POR COLUMNA")
print("=" * 60)
print(df.isnull().sum())

print("\n" + "=" * 60)
print("ESTADÍSTICAS DESCRIPTIVAS")
print("=" * 60)
print(df.describe())



Exploracion del dataset sS&p 500 Financials

📊 dimensiones: 503 filas, 14 columnas

📋 Columnas:
['Symbol', 'Name', 'Sector', 'Price', 'Price/Earnings', 'Dividend Yield', 'Earnings/Share', '52 Week Low', '52 Week High', 'Market Cap', 'EBITDA', 'Price/Sales', 'Price/Book', 'SEC Filings']

Tipos de datos
Symbol                str
Name                  str
Sector                str
Price             float64
Price/Earnings    float64
Dividend Yield    float64
Earnings/Share    float64
52 Week Low       float64
52 Week High      float64
Market Cap        float64
EBITDA            float64
Price/Sales       float64
Price/Book        float64
SEC Filings           str
dtype: object

VALORES FALTANTES (NaN) POR COLUMNA
Symbol              0
Name                0
Sector              0
Price              17
Price/Earnings     48
Dividend Yield    104
Earnings/Share     17
52 Week Low        17
52 Week High       17
Market Cap         17
EBITDA             43
Price/Sales        17
Price/Book        

In [18]:

df.isnull().sum()
# Mostrar SOLO columnas que tienen valores faltantes
nan_por_columna = df.isnull().sum()
nan_por_columna[nan_por_columna > 0]


Price              17
Price/Earnings     48
Dividend Yield    104
Earnings/Share     17
52 Week Low        17
52 Week High       17
Market Cap         17
EBITDA             43
Price/Sales        17
Price/Book         17
dtype: int64

In [19]:
# ============================================
# LIMPIEZA 1: Eliminar filas sin precio
# ============================================

print(f"Antes de limpiar: {df.shape[0]} filas")

# Eliminar filas donde 'Price' es NaN
# (Sin precio = no podemos analizar la empresa)
df_limpio = df.dropna(subset=['Price'])

print(f"Después de limpiar: {df_limpio.shape[0]} filas")
print(f"Eliminadas: {df.shape[0] - df_limpio.shape[0]} filas")

# Verificar que ya no hay NaN en Price
print(f"\nNaN en Price después: {df_limpio['Price'].isnull().sum()}")

Antes de limpiar: 503 filas
Después de limpiar: 486 filas
Eliminadas: 17 filas

NaN en Price después: 0


In [20]:
# Ver NaN restantes después de eliminar filas sin precio
nan_restantes = df_limpio.isnull().sum()
nan_restantes[nan_restantes > 0]

Price/Earnings    31
Dividend Yield    87
EBITDA            26
dtype: int64

In [21]:
# ============================================
# LIMPIEZA 2: Rellenar Dividend Yield con 0
# ============================================

print(f"NaN en Dividend Yield ANTES: {df_limpio['Dividend Yield'].isnull().sum()}")

# Rellenar NaN con 0 (empresas que no pagan dividendos)
df_limpio['Dividend Yield'] = df_limpio['Dividend Yield'].fillna(0)

print(f"NaN en Dividend Yield DESPUÉS: {df_limpio['Dividend Yield'].isnull().sum()}")
print(f"\nEmpresas que NO pagan dividendos: {(df_limpio['Dividend Yield'] == 0).sum()}")

NaN en Dividend Yield ANTES: 87
NaN en Dividend Yield DESPUÉS: 0

Empresas que NO pagan dividendos: 87


In [22]:
# ============================================
# VERIFICACIÓN: ¿Qué NaN quedan?
# ============================================

nan_restantes = df_limpio.isnull().sum()
print("Columnas con NaN restantes:")
print(nan_restantes[nan_restantes > 0])

print(f"\nTotal de filas en dataset limpio: {df_limpio.shape[0]}")
print(f"Total de columnas: {df_limpio.shape[1]}")

Columnas con NaN restantes:
Price/Earnings    31
EBITDA            26
dtype: int64

Total de filas en dataset limpio: 486
Total de columnas: 14


In [23]:
# ============================================
# LIMPIEZA 3: Verificar duplicados
# ============================================

print(f"Filas duplicadas (todas las columnas iguales): {df_limpio.duplicated().sum()}")

# Verificar duplicados por Symbol (ticker único)
simbolos_duplicados = df_limpio[df_limpio.duplicated(subset=['Symbol'], keep=False)]
print(f"\nEmpresas con Symbol duplicado: {len(simbolos_duplicados)}")

if len(simbolos_duplicados) > 0:
    print("\nSímbolos duplicados:")
    print(simbolos_duplicados[['Symbol', 'Name']].sort_values('Symbol'))
else:
    print("\n✅ No hay duplicados. Cada empresa aparece una sola vez.")

Filas duplicadas (todas las columnas iguales): 0

Empresas con Symbol duplicado: 0

✅ No hay duplicados. Cada empresa aparece una sola vez.


In [24]:
# ============================================
# LIMPIEZA 4: Detectar outliers
# ============================================

print("=" * 60)
print("DETECCIÓN DE OUTLIERS")
print("=" * 60)

# Ver estadísticas de columnas numéricas clave
print("\n📊 Estadísticas de precios y ratios:")
print(df_limpio[['Price', 'Price/Earnings', 'Dividend Yield', 'Market Cap']].describe())

# Empresas con precio extremadamente alto (> $1000)
print("\n🔴 Empresas con precio > $1000 (posibles outliers):")
outliers_precio = df_limpio[df_limpio['Price'] > 1000][['Symbol', 'Name', 'Sector', 'Price']]
print(outliers_precio.sort_values('Price', ascending=False))

# Empresas con P/E extremadamente alto (> 100, posible sobrevaloración o error)
print("\n🟡 Empresas con P/E > 100 (posible sobrevaloración):")
outliers_pe = df_limpio[df_limpio['Price/Earnings'] > 100][['Symbol', 'Name', 'Sector', 'Price/Earnings']]
print(outliers_pe.sort_values('Price/Earnings', ascending=False).head(10))

DETECCIÓN DE OUTLIERS

📊 Estadísticas de precios y ratios:
             Price  Price/Earnings  Dividend Yield    Market Cap
count   486.000000      455.000000      486.000000  4.860000e+02
mean    231.545617       41.137141        0.017621  1.478906e+11
std     377.620154      107.241567        0.015672  4.900115e+11
min       1.700000        0.006677        0.000000  5.634492e+06
25%      73.715000       17.705540        0.005125  2.102989e+10
50%     142.975000       24.578617        0.013900  4.149905e+10
75%     279.677500       35.770208        0.027125  9.610322e+10
max    6307.930000     1701.058700        0.078400  5.453600e+12

🔴 Empresas con precio > $1000 (posibles outliers):
    Symbol                      Name  \
351    NVR                 NVR, Inc.   
51     AZO                  AutoZone   
316    MTD            Mettler Toledo   
327   MPWR  Monolithic Power Systems   
480    GWW            W. W. Grainger   
451    TDG           TransDigm Group   
290    LLY              

In [25]:
# ============================================
# LIMPIEZA 5: Feature Engineering
# ============================================

# 1. Categoría: ¿Paga dividendos?
df_limpio['Paga_Dividendos'] = df_limpio['Dividend Yield'].apply(lambda x: 'Sí' if x > 0 else 'No')

# 2. Categoría: Tamaño de empresa por Market Cap
def clasificar_tamano(market_cap):
    if market_cap >= 200e9:      # $200B+
        return 'Large Cap (Mega)'
    elif market_cap >= 10e9:     # $10B - $200B
        return 'Large Cap'
    elif market_cap >= 2e9:      # $2B - $10B
        return 'Mid Cap'
    else:
        return 'Small Cap'

df_limpio['Tamanio_Empresa'] = df_limpio['Market Cap'].apply(clasificar_tamano)

# 3. Categoría: Valuación por P/E (solo empresas con P/E definido)
def clasificar_valuacion(pe):
    if pd.isna(pe):
        return 'Sin P/E (Pérdidas)'
    elif pe < 15:
        return 'Subvalorada'
    elif pe < 30:
        return 'Neutral'
    else:
        return 'Sobrevalorada'

df_limpio['Valuacion_PE'] = df_limpio['Price/Earnings'].apply(clasificar_valuacion)

# Verificar las nuevas columnas
print("=" * 60)
print("NUEVAS COLUMNAS CREADAS")
print("=" * 60)
print(df_limpio[['Symbol', 'Name', 'Paga_Dividendos', 'Tamanio_Empresa', 'Valuacion_PE']].head(15))

NUEVAS COLUMNAS CREADAS
   Symbol                             Name Paga_Dividendos   Tamanio_Empresa  \
0     MMM                               3M              Sí         Large Cap   
1     AOS                      A. O. Smith              Sí           Mid Cap   
2     ABT              Abbott Laboratories              Sí         Large Cap   
3    ABBV                           AbbVie              Sí  Large Cap (Mega)   
4     ACN                        Accenture              Sí         Large Cap   
5    ADBE                       Adobe Inc.              No         Large Cap   
6     AMD           Advanced Micro Devices              No  Large Cap (Mega)   
7     AES                  AES Corporation              Sí         Large Cap   
8     AFL                            Aflac              Sí         Large Cap   
9       A             Agilent Technologies              Sí         Large Cap   
10    APD                     Air Products              Sí         Large Cap   
11   ABNB       

In [26]:
# ============================================
# PASO FINAL: Guardar dataset limpio
# ============================================

# Guardar como CSV limpio
df_limpio.to_csv("sp500_financials_limpio.csv", index=False, encoding="utf-8")
print("✅ Dataset limpio guardado: sp500_financials_limpio.csv")

# ============================================
# RESUMEN DE LIMPIEZA
# ============================================

print("\n" + "=" * 60)
print("📊 RESUMEN DE LIMPIEZA - P5")
print("=" * 60)

print(f"\n🔹 Dataset original: 503 filas × 14 columnas")
print(f"🔹 Dataset limpio: {df_limpio.shape[0]} filas × {df_limpio.shape[1]} columnas")
print(f"🔹 Filas eliminadas: {503 - df_limpio.shape[0]} (sin precio)")
print(f"🔹 Columnas nuevas: 3 (Paga_Dividendos, Tamanio_Empresa, Valuacion_PE)")

print(f"\n🔹 NaN tratados:")
print(f"   • Dividend Yield: 87 → rellenados con 0")
print(f"   • Price/Earnings: 31 → dejados como NaN (pérdidas reales)")
print(f"   • EBITDA: 26 → dejados como NaN")

print(f"\n🔹 Duplicados: 0")
print(f"🔹 Outliers: Documentados (precios altos reales, no errores)")

print(f"\n🔹 Empresas que NO pagan dividendos: {(df_limpio['Paga_Dividendos'] == 'No').sum()}")
print(f"🔹 Mega Caps: {(df_limpio['Tamanio_Empresa'] == 'Large Cap (Mega)').sum()}")
print(f"🔹 Empresas con pérdidas: {(df_limpio['Valuacion_PE'] == 'Sin P/E (Pérdidas)').sum()}")

✅ Dataset limpio guardado: sp500_financials_limpio.csv

📊 RESUMEN DE LIMPIEZA - P5

🔹 Dataset original: 503 filas × 14 columnas
🔹 Dataset limpio: 486 filas × 17 columnas
🔹 Filas eliminadas: 17 (sin precio)
🔹 Columnas nuevas: 3 (Paga_Dividendos, Tamanio_Empresa, Valuacion_PE)

🔹 NaN tratados:
   • Dividend Yield: 87 → rellenados con 0
   • Price/Earnings: 31 → dejados como NaN (pérdidas reales)
   • EBITDA: 26 → dejados como NaN

🔹 Duplicados: 0
🔹 Outliers: Documentados (precios altos reales, no errores)

🔹 Empresas que NO pagan dividendos: 87
🔹 Mega Caps: 57
🔹 Empresas con pérdidas: 31
